# Modelo de Detección de Fraude Financiero
## Caso Integrador Final — Programa de Especialización en Credit Scoring con Python

**Escenario:** Una institución financiera requiere fortalecer su sistema de gestión de riesgo
operacional mediante un modelo de ML capaz de identificar transacciones anómalas (fraude).

**Variable objetivo:** `fraud` — binaria (0 = legítima, 1 = fraude), tasa de incidencia ~2%.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve,
                              precision_recall_curve, average_precision_score,
                              f1_score, recall_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

RANDOM_STATE = 42
TEST_SIZE    = 0.20
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerias cargadas correctamente.")

In [ ]:
df = pd.read_csv("../data/base.csv")
print(f"Shape: {df.shape}")
print(f"
Columnas: {df.columns.tolist()}")
df.head()

## Módulo 1: Análisis Exploratorio de Datos (EDA)

> "Un buen modelo no nace en el algoritmo, nace en el EDA." — Instructor del programa.

El EDA es la base diagnóstica que justifica todas las decisiones técnicas posteriores.
Se analiza la calidad de los datos, el desbalance de clases, la distribución de variables
y el comportamiento diferenciado entre transacciones legítimas y fraudulentas.

In [ ]:
print("=== ESTADÍSTICOS DESCRIPTIVOS ===")
display(df.describe(percentiles=[.25, .50, .75, .90, .99]).round(2))

print("
=== VALORES NULOS ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else "No se encontraron valores nulos.")
print(f"
Tipos de datos:
{df.dtypes}")

In [ ]:
conteo     = df['fraud'].value_counts()
pct_fraude = df['fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Legítima (0)', 'Fraude (1)'], conteo.values,
            color=['steelblue', 'tomato'])
axes[0].set_title('Distribución de la variable objetivo')
axes[0].set_ylabel('Cantidad de transacciones')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(conteo.values, labels=['Legítima', 'Fraude'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporción')

plt.suptitle(f'Desbalance de clases — Tasa de fraude: {pct_fraude:.2f}%',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f"
→ {conteo[0]:,} legítimas vs {conteo[1]:,} fraudes ({pct_fraude:.2f}%)")
print("→ Estrategia requerida: SMOTE + métricas Recall/F1 (no Accuracy global)")